To run MolMIM you need to have selected the following options on the REANNZ Open OnDemand Menu:

<div style="float: left;">
    
| Option | Value |
| :---------- | :--------- |
| Cluster | Slurm HPC |  
| Project Code | uoa04517 | 
| GPU | L4 or H100 |

</div>

In [1]:
API_KEY= None  # papermill overrides this when run non-interactively

You will also need an API key from NGC.  If you do not have one, the script 'check_molmim_api_key.sh' should ensure you have a valid, secure key saved at ~/ngc/ngc_api_key.molmim

In [2]:
import os
if not API_KEY:
    with open(os.path.expanduser("~/.ngc/ngc_api_key.molmim")) as f:
        API_KEY = f.read().strip()
os.environ["NGC_API_KEY"] = API_KEY
os.environ["CACHEDIR"] = "/nesi/nobackup/uoa04517/cache"
os.environ["LOCAL_NVS_CACHE"] = "/nesi/nobackup/uoa04517/cache"

In [3]:
#Start Molmim API server
import subprocess, time

log_path = "server.log"
proc = subprocess.Popen(
    "apptainer run --env NVIDIA_VISIBLE_DEVICES=0 --env NGC_API_KEY=$NGC_API_KEY "
    "--bind $LOCAL_NVS_CACHE:/home/nvs/.cache --bind $CACHEDIR:/tmp --writable-tmpfs --nv "
    f"molmim.sif /usr/local/bin/start_server > {log_path} 2>&1",
    shell=True,
)

time.sleep(2)  # assume server.log exists by now
f = open(log_path)

while True:
    line = f.readline()
    if line:
        if "Uvicorn running" in line:
            break
    elif proc.poll() is not None:
        raise RuntimeError(f"Server exited early (code {proc.returncode}, check {log_path})")
    else:
        time.sleep(1)

print("MolMIM server ready.")

RuntimeError: Server exited early (code 1, check server.log)

Start MolMIM

In [ ]:
import os
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot
cwd= os.getcwd()
print (cwd)

In [ ]:
os.chdir("/nesi/project/uoa04517/1. Molecules_generated_MolMIM-GenMol/MolMIM_12K")
cwd=os.getcwd()
print(cwd)

In [ ]:
invoke_url = "http://localhost:8000/generate"

headers = {
    'accept': 'application/json',
    'Content-Type': 'application/json'
}

payload = {
  "algorithm": "CMA-ES",
  "num_molecules": 30,
  "property_name": "QED",
  "minimize": False,
  "min_similarity": 0.3,
  "particles": 30,
  "iterations": 20,
  "smi": "O=C1OC2=CC=C(NC(C3=CC=CC=C3OCC4=CC=CC(C5=CC([C@H](N)CO6)=C6C=C5)=C4)C(O)=O)C=C2N1"
}

response_appended = [] #Initialise, so that the for loop values are appended appended values iteratively feeds back
                        #And builds into this list
for i in range (400):
    session = requests.Session()

    response = session.post(invoke_url, headers=headers, json=payload)
    
    response.raise_for_status()
    response_body = response.json()
    smiles= response_body.get("generated", [])
    response_appended.extend(smiles)
    print(response_body)
    
print(response_appended)
df=pd.DataFrame(response_appended)

df.to_csv("molecule_3_MolMIM_Analogues_12K.csv")

In [ ]:
import requests
import json

url = "http://localhost:8000/embedding"

headers = {
    'accept': 'application/json',
    'Content-Type': 'application/json'
}

data = json.dumps({"sequences": ["CC(Cc1ccc(cc1)C(C(=O)O)C)C"]})

response = requests.post(url, headers=headers, data=data)

print(response.text)